# MaNGA HEALPix collision check
Determine the minimum `highest_healpix_order` needed for 1 row per pixel.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import h5py
import healpy as hp
import numpy as np
from astropy.io import fits

BASE = Path('/mnt/ceph/users/polymathic/MultimodalUniverse/manga/manga')
HIST = '/mnt/ceph/users/polymathic/manga-staging-tmp/mmu_manga/intermediate/mmu_manga/intermediate/row_count_mapping_histogram.npz'
FITS = '/mnt/home/thehir/tmp-catalog-downloads/MaNGA_targets_extNSA_tiled_ancillary.fits'

## FITS target catalog: collision distribution by healpix order

In [ ]:
with fits.open(FITS) as hdul:
    data = hdul[1].data
    print(hdul[1].columns.names[:20])
ra = np.asarray(data['CATALOG_RA'], dtype=np.float64)
dec = np.asarray(data['CATALOG_DEC'], dtype=np.float64)
print(f'n={len(ra)}')

['CATALOG_RA', 'CATALOG_DEC', 'NSA_Z', 'NSA_ZDIST', 'NSA_ELPETRO_MASS', 'NSA_ELPETRO_ABSMAG', 'NSA_ELPETRO_AMIVAR', 'NSA_ELPETRO_FLUX', 'NSA_ELPETRO_FLUX_IVAR', 'NSA_ELPETRO_TH50_R', 'NSA_ELPETRO_TH50', 'NSA_ELPETRO_PHI', 'NSA_ELPETRO_BA', 'NSA_SERSIC_MASS', 'NSA_SERSIC_ABSMAG', 'NSA_SERSIC_AMIVAR', 'NSA_SERSIC_FLUX', 'NSA_SERSIC_FLUX_IVAR', 'NSA_SERSIC_TH50', 'NSA_SERSIC_PHI']
n=43068


In [ ]:
for order in range(5, 30):
    nside = 2 ** order
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=True)
    _, counts = np.unique(pix, return_counts=True)
    print(f'order={order:2d} max_per_pixel={counts.max():5d} n_collisions={(counts>1).sum()}')

order= 5 max_per_pixel=  258 n_collisions=2568
order= 6 max_per_pixel=  135 n_collisions=7837
order= 7 max_per_pixel=   87 n_collisions=10314
order= 8 max_per_pixel=   33 n_collisions=6972
order= 9 max_per_pixel=   10 n_collisions=3756
order=10 max_per_pixel=    5 n_collisions=1763
order=11 max_per_pixel=    4 n_collisions=701
order=12 max_per_pixel=    3 n_collisions=262
order=13 max_per_pixel=    2 n_collisions=118
order=14 max_per_pixel=    2 n_collisions=48
order=15 max_per_pixel=    2 n_collisions=23
order=16 max_per_pixel=    2 n_collisions=7
order=17 max_per_pixel=    2 n_collisions=6
order=18 max_per_pixel=    2 n_collisions=6
order=19 max_per_pixel=    2 n_collisions=6
order=20 max_per_pixel=    2 n_collisions=6
order=21 max_per_pixel=    2 n_collisions=6
order=22 max_per_pixel=    2 n_collisions=6
order=23 max_per_pixel=    2 n_collisions=6
order=24 max_per_pixel=    2 n_collisions=6
order=25 max_per_pixel=    2 n_collisions=6
order=26 max_per_pixel=    2 n_collisions=6
order

## HDF5 ingest data: order-10 histogram from staging

In [ ]:
with open(HIST, 'rb') as f:
    full = np.frombuffer(f.read(), dtype=np.int64)

hist_order = int(np.log2(np.sqrt(len(full) / 12)))
print(f'histogram order={hist_order}, total rows={full.sum()}, nonempty pixels={(full>0).sum()}, max per pixel={full.max()}')
for k in range(1, full.max() + 1):
    n = int((full == k).sum())
    if n:
        print(f'  pixels with {k} rows: {n}')

histogram order=10, total rows=10735, nonempty pixels=9973, max per pixel=62
  pixels with 1 rows: 9598
  pixels with 2 rows: 326
  pixels with 3 rows: 28
  pixels with 4 rows: 2
  pixels with 5 rows: 6
  pixels with 6 rows: 2
  pixels with 8 rows: 2
  pixels with 21 rows: 1
  pixels with 23 rows: 1
  pixels with 27 rows: 1
  pixels with 31 rows: 1
  pixels with 33 rows: 1
  pixels with 34 rows: 1
  pixels with 49 rows: 1
  pixels with 55 rows: 1
  pixels with 62 rows: 1


## Locate the densest HDF5 pixel and inspect it

In [ ]:
pat = re.compile(r'healpix=(\d+)$')
existing = sorted(int(m.group(1)) for d in BASE.iterdir() if (m := pat.match(d.name)))
print(f'{len(existing)} healpix dirs; max pixel number = {max(existing)}')

mmu_order = next(o for o in range(0, 15) if max(existing) < 12 * 4 ** o)
print(f'MMU path order = {mmu_order}')

nside_mmu = 2 ** mmu_order
densest_pixel10 = int(np.argmax(full))
shift = 2 * (hist_order - mmu_order)
mmu_pixel = densest_pixel10 >> shift
print(f'densest order-{hist_order} pixel={densest_pixel10} count={int(full[densest_pixel10])}')
print(f'densest MMU pixel at order {mmu_order}: healpix={mmu_pixel}')

dir_path = BASE / f'healpix={mmu_pixel}'
files = sorted(dir_path.glob('*.hdf5'))
print(f'files in dir: {[f.name for f in files]}')

449 healpix dirs; max pixel number = 3071
MMU path order = 4
densest order-10 pixel=897653 count=62
densest MMU pixel at order 4: healpix=219
files in dir: ['0001-of-0006.hdf5', '0002-of-0006.hdf5', '0003-of-0006.hdf5', '0004-of-0006.hdf5', '0005-of-0006.hdf5', '0006-of-0006.hdf5']


## Do the 6 bit-identical FITS pairs appear in the HDF5 data?

In [ ]:
coord_counts = Counter(zip(ra.tolist(), dec.tolist()))
dup_coords = [(r, d, n) for (r, d), n in coord_counts.items() if n > 1]
print(f'bit-identical (ra, dec) groups in FITS: {len(dup_coords)}')
for r, d, n in dup_coords:
    print(f'  ra={r:.10f} dec={d:.10f} count={n}')

dup_mmu_pixels = sorted({int(hp.ang2pix(nside_mmu, r, d, lonlat=True, nest=True)) for r, d, _ in dup_coords})
print(f'\nMMU order-{mmu_order} pixels containing these duplicates: {dup_mmu_pixels}')

bit-identical (ra, dec) groups in FITS: 6
  ra=160.7922398611 dec=39.0386577270 count=2
  ra=239.3400304118 dec=25.7886520911 count=2
  ra=224.1506824795 dec=9.3697228585 count=2
  ra=136.0330750675 dec=17.4411987193 count=2
  ra=233.4228060782 dec=28.1454829045 count=2
  ra=150.5889333330 dec=1.1492666667 count=2

MMU order-4 pixels containing these duplicates: [271, 344, 521, 541, 565, 1703]


In [ ]:
TOL = 1e-9

for dup_pixel in dup_mmu_pixels:
    d = BASE / f'healpix={dup_pixel}'
    dfiles = sorted(d.glob('*.hdf5')) if d.exists() else []
    print(f'\n=== MMU healpix={dup_pixel} ({len(dfiles)} files) ===')
    if not dfiles:
        print('  directory missing or empty')
        continue
    groups = []
    for fpath in dfiles:
        with h5py.File(fpath, 'r') as hf:
            for name in hf.keys():
                g = hf[name]
                ra_g = float(g['ra'][()] if 'ra' in g else g['RA'][()])
                dec_g = float(g['dec'][()] if 'dec' in g else g['DEC'][()])
                oid = g['object_id'][()]
                if isinstance(oid, bytes):
                    oid = oid.decode()
                groups.append((fpath.name, name, oid, ra_g, dec_g))
    for r_fits, d_fits, n in dup_coords:
        if int(hp.ang2pix(nside_mmu, r_fits, d_fits, lonlat=True, nest=True)) != dup_pixel:
            continue
        print(f'  FITS duplicate ra={r_fits:.10f} dec={d_fits:.10f} (count={n}):')
        matches = [g for g in groups if abs(g[3] - r_fits) < TOL and abs(g[4] - d_fits) < TOL]
        if not matches:
            print('    no HDF5 groups match this coordinate')
        for fname, gname, oid, ra_g, dec_g in matches:
            print(f'    file={fname} group={gname} object_id={oid} ra={ra_g:.10f} dec={dec_g:.10f}')


=== MMU healpix=271 (1 files) ===
  FITS duplicate ra=136.0330750675 dec=17.4411987193 (count=2):
    no HDF5 groups match this coordinate

=== MMU healpix=344 (0 files) ===
  directory missing or empty

=== MMU healpix=521 (0 files) ===
  directory missing or empty

=== MMU healpix=541 (1 files) ===
  FITS duplicate ra=239.3400304118 dec=25.7886520911 (count=2):
    no HDF5 groups match this coordinate

=== MMU healpix=565 (1 files) ===
  FITS duplicate ra=233.4228060782 dec=28.1454829045 (count=2):
    no HDF5 groups match this coordinate

=== MMU healpix=1703 (1 files) ===
  FITS duplicate ra=150.5889333330 dec=1.1492666667 (count=2):
    no HDF5 groups match this coordinate


## Densest MMU pixel: are the 62 rows at unique (ra, dec, object_id)?

In [ ]:
rows = []
for fpath in files:
    with h5py.File(fpath, 'r') as hf:
        for name in hf.keys():
            g = hf[name]
            ra_g = float(g['ra'][()] if 'ra' in g else g['RA'][()])
            dec_g = float(g['dec'][()] if 'dec' in g else g['DEC'][()])
            oid = g['object_id'][()]
            if isinstance(oid, bytes):
                oid = oid.decode()
            rows.append((fpath.name, name, str(oid), ra_g, dec_g))

print(f'total rows in healpix={mmu_pixel}: {len(rows)}')
print(f'unique object_id: {len({r[2] for r in rows})}')
print(f'unique (ra, dec) (exact): {len({(r[3], r[4]) for r in rows})}')

coord_counts = Counter((r[3], r[4]) for r in rows)
dup_in_pixel = [(r, d, n) for (r, d), n in coord_counts.items() if n > 1]
print(f'bit-identical (ra, dec) groups within this pixel: {len(dup_in_pixel)}')
for r, d, n in dup_in_pixel:
    members = [x for x in rows if x[3] == r and x[4] == d]
    print(f'  ra={r:.10f} dec={d:.10f} count={n}')
    for fname, gname, oid, _, _ in members:
        print(f'    file={fname} group={gname} object_id={oid}')

ras = np.array([r[3] for r in rows])
decs = np.array([r[4] for r in rows])
for order in range(10, 30):
    nside = 2 ** order
    pix = hp.ang2pix(nside, ras, decs, lonlat=True, nest=True)
    _, counts = np.unique(pix, return_counts=True)
    print(f'order={order:2d} max_per_pixel={counts.max():3d} n_collisions={(counts>1).sum()}')

total rows in healpix=219: 345
unique object_id: 345
unique (ra, dec) (exact): 342
bit-identical (ra, dec) groups within this pixel: 2
  ra=56.7023380000 dec=68.0966250000 count=2
    file=0001-of-0006.hdf5 group=10141-12701 object_id=10141-12701
    file=0006-of-0006.hdf5 group=9675-12703 object_id=9675-12703
  ra=56.1948420000 dec=68.4531780000 count=3
    file=0006-of-0006.hdf5 group=9673-6102 object_id=9673-6102
    file=0006-of-0006.hdf5 group=9674-6102 object_id=9674-6102
    file=0006-of-0006.hdf5 group=9675-6102 object_id=9675-6102
order=10 max_per_pixel= 62 n_collisions=11
order=11 max_per_pixel= 22 n_collisions=28
order=12 max_per_pixel=  6 n_collisions=90
order=13 max_per_pixel=  3 n_collisions=60
order=14 max_per_pixel=  3 n_collisions=7
order=15 max_per_pixel=  3 n_collisions=4
order=16 max_per_pixel=  3 n_collisions=4
order=17 max_per_pixel=  3 n_collisions=3
order=18 max_per_pixel=  3 n_collisions=2
order=19 max_per_pixel=  3 n_collisions=2
order=20 max_per_pixel=  3 n_c

## Inspect FITS columns for target/selection flags
Do the catalog's duplicated rows (and absent FITS rows) have flags distinguishing "would be ingested" from "skipped"?

In [ ]:
with fits.open(FITS) as hdul:
    cols = hdul[1].columns
    all_names = cols.names
print(f'total columns: {len(all_names)}')
flag_like = [n for n in all_names if any(k in n.upper() for k in
    ['TARGET', 'FLAG', 'MASK', 'MANGAID', 'IFU', 'PLATE', 'SAMPLE', 'PRIMARY', 'SECONDARY', 'COLOR', 'ANCILL'])]
print(f'flag-like columns ({len(flag_like)}):')
for n in flag_like:
    print(f'  {n}')

total columns: 70
flag-like columns (15):
  MANGA_TARGET1
  MANGAID
  RANFLAG
  IFUTARGETSIZE
  IFUDESIGNSIZE
  IFUDESIGNWRONGSIZE
  IFU_RA
  IFU_DEC
  BADPHOTFLAG
  STARFLAG
  OBSFLAG
  IFUDESIGNSIZEMAIN
  IFUMINSIZEANC
  IFUTARGSIZEANC
  MANGA_TARGET3


In [ ]:
with fits.open(FITS) as hdul:
    d = hdul[1].data
    inspect_cols = [c for c in flag_like if c in d.names]
    idx_sets = []
    for r_fits, d_fits, n in dup_coords:
        idx = np.where((d['CATALOG_RA'] == r_fits) & (d['CATALOG_DEC'] == d_fits))[0]
        idx_sets.append((r_fits, d_fits, idx))

    for r_fits, d_fits, idx in idx_sets:
        print(f'\nra={r_fits:.6f} dec={d_fits:.6f} rows={list(idx)}')
        for col in inspect_cols:
            vals = [d[col][i] for i in idx]
            if any(v != vals[0] for v in vals[1:]):
                marker = '  [DIFFER]'
            else:
                marker = ''
            print(f'  {col}: {vals}{marker}')


ra=160.792240 dec=39.038658 rows=[np.int64(16971), np.int64(42464)]
  MANGA_TARGET1: [np.int32(1040), np.int32(0)]  [DIFFER]
  MANGAID: ['1-275734', '42-117']  [DIFFER]
  RANFLAG: [np.uint8(1), np.uint8(25)]  [DIFFER]
  IFUTARGETSIZE: [np.int16(91), np.int16(-999)]  [DIFFER]
  IFUDESIGNSIZE: [np.int16(127), np.int16(61)]  [DIFFER]
  IFUDESIGNWRONGSIZE: [np.int16(0), np.int16(0)]
  IFU_RA: [np.float64(160.79223986107647), np.float64(160.79223986107647)]
  IFU_DEC: [np.float64(39.03865772703269), np.float64(39.03865772703269)]
  BADPHOTFLAG: [np.uint8(0), np.uint8(0)]
  STARFLAG: [np.uint8(0), np.uint8(0)]
  OBSFLAG: [np.uint8(0), np.uint8(0)]
  IFUDESIGNSIZEMAIN: [np.int16(127), np.int16(-999)]  [DIFFER]
  IFUMINSIZEANC: [np.int16(61), np.int16(61)]
  IFUTARGSIZEANC: [np.int16(127), np.int16(127)]
  MANGA_TARGET3: [np.int32(8388608), np.int32(8192)]  [DIFFER]

ra=239.340030 dec=25.788652 rows=[np.int64(19251), np.int64(42595)]
  MANGA_TARGET1: [np.int32(1168), np.int32(0)]  [DIFFER]
  